# Phase 4 · Analysis of the canonical v2 results (R1 revision)

Input: `output/phase3/xfi_<model>/results_main.csv`, `results_base.csv` (canonical v2 run, D-04) and the classifier outputs of Phase 2.
Models: **A** (submitted, ART-008; primary), **B** (ImageNet-pretrained, pos_weight; replication), and, when present, **Anorm** and **Arep** (v1 recipe with and without ImageNet normalisation).

| Section | Request | Tables | Figure |
|---|---|---|---|
| 1 | R1-a, R1-b, R1-d, R2-M1, R2-M3: classifiers, thresholds, calibration, normalisation | R1, R1b, R1c | RF1, RF2 |
| 2 | A-03, D-15: Grad-CAM vs Eigen-CAM, patient as unit, LMM | R2, R2b | RF3 |
| 3 | A-13: kinetics and slopes, with the cross-patient reference | R3, R4 | RF4 |
| 4 | R1-c: localization by prediction correctness and under perturbation | R5, R6 | RF5 |
| 5 | R2-M2: absolute-threshold false activation | R7 | RF6 |
| 6 | A-04, A-05, D-17: validity (input PSNR, localization change, prediction flips) and component ablation | R8, R9 | RF7 |
| 7 | D-11: Eigen-CAM sign ablation | R10 | RF8 |
| 8 | D-14: baselines | R11 | RF9 |
| 9 | A-11: prediction flips | R12 | RF10 |
| 10 | R1-V1, R1-V2: consistency across classifiers; numbers for the manuscript | R13, numbers_v2.json | |

All CIs: patient bootstrap, B = 2000, percentile 95%. Tables go to `output/phase4/`, figures to `fig/phase4/` (PNG + PDF 300 dpi + CSV of the plotted data).

## Configuration cell

In [ ]:
import sys, json, hashlib, time
from pathlib import Path
import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from r1lib import style as S, stats as ST
S.apply_style()
CFG = yaml.safe_load(open(ROOT / "configs" / "config.yaml"))
SEED, B = CFG["project"]["seed"], CFG["metrics"]["bootstrap_B"]
OUT = ROOT / "output" / "phase4"; OUT.mkdir(parents=True, exist_ok=True)
FIG = ROOT / "fig" / "phase4"; FIG.mkdir(parents=True, exist_ok=True)
MODEL_ORDER = ["A", "Arep", "Anorm", "B"]
MODELS = [m for m in MODEL_ORDER if (ROOT / "output" / "phase3" / f"xfi_{m}" / "results_main.csv").exists()]
MODEL_LABELS = {"A": "Model A (submitted)", "Arep": "Model A-rep", "Anorm": "Model A-norm", "B": "Model B (pretrained)"}
MODEL_LS = {"A": "-", "Arep": "--", "Anorm": ":", "B": "-."}
MODEL_INK = {"A": "#0b0b0b", "Arep": "#52514e", "Anorm": "#52514e", "B": "#0b0b0b"}
METHODS = ["GradCAM", "EigenCAM"]
TYPES, LEVELS = S.TYPES, [1, 2, 3]
PRODUCED, NUM = [], {}            # files written; numbers for the manuscript

def sha256(p):
    h = hashlib.sha256(); h.update(Path(p).read_bytes()); return h.hexdigest()
def save_table(df, name):
    p = OUT / f"{name}.csv"; df.to_csv(p, index=False); PRODUCED.append(p); return df
def save_fig(fig, stem, data):
    PRODUCED.extend(S.save_figure(fig, stem, data, FIG))
def ci(x, stat=np.mean):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    return ST.boot_ci_stat(x, stat, B=B, seed=SEED) if len(x) else (np.nan, np.nan, np.nan)
print("models:", MODELS)

## Load

One row per patient x method x condition in `main`; baselines in `base`. `grp` is the prediction-correctness group of the clean image under each model's own operating threshold (A: 0.5; others: validation Youden threshold, D-30).

In [ ]:
main, base, man = {}, {}, {}
for M in MODELS:
    d = ROOT / "output" / "phase3" / f"xfi_{M}"
    man[M] = json.loads((d / "manifest.json").read_text())
    m = pd.read_csv(d / "results_main.csv"); b = pd.read_csv(d / "results_base.csv")
    assert sha256(d / "results_main.csv") == man[M]["results_main_sha256"], f"{M}: results_main hash mismatch"
    m["grp"] = np.select([(m.Label == 1) & (m.pred_clean == 1), (m.Label == 1) & (m.pred_clean == 0),
                          (m.Label == 0) & (m.pred_clean == 0)], ["TP", "FN", "TN"], "FP")
    m["d_iou"] = m.iou_rel90_c - m.iou_rel90_n; m["abs_d_iou"] = m.d_iou.abs()
    main[M], base[M] = m, b
    print(f"{M}: rows {len(m)} / {len(b)}  thr {man[M]['threshold']:.4f}  groups {man[M]['groups']}  git {man[M]['git']}")

## 1 · Classifiers (R1-a, R1-b, R1-d, R2-M1, R2-M3)

**R1** lists, for every model on the same 5,337 test patients: AUC, average precision, ECE, Brier, and accuracy, precision, recall, specificity, F1 at 0.5 and at the model's operating threshold (A: 0.5, as in v1; the others: Youden threshold chosen on the validation split). Model A was trained on the full v1 training set, so it has no separate validation split.
**R1b**: paired bootstrap differences in AUC and ECE (A-norm vs A-rep isolates ImageNet normalisation; B vs A is the overall change).
**R1c**: post-hoc recalibration fitted on the validation split, temperature scaling (1 parameter) and Platt scaling (slope and intercept on the logit). Reporting only: the classifier is not changed, and Grad-CAM maps are invariant to a positive affine transform of the logit. A pure temperature cannot remove the upward shift that pos_weight introduces; Platt scaling can.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve
from scipy.optimize import minimize_scalar
labels = main["A"].drop_duplicates("Patient").set_index("Patient")["Label"] if "A" in main else None
probs, vals, thr = {}, {}, {}
probs["A"] = main["A"].drop_duplicates("Patient").set_index("Patient")["prob_clean"]; thr["A"] = 0.5
for M in ["Arep", "Anorm", "B"]:
    d = ROOT / "output" / "phase2" / f"train_{M}"
    if (d / "test_probs.csv").exists():
        t = pd.read_csv(d / "test_probs.csv").set_index("patientId"); probs[M] = t["prob"]
        vals[M] = pd.read_csv(d / "val_probs.csv"); thr[M] = json.loads((d / "summary.json").read_text())["val_youden_threshold"]
CLS = [m for m in MODEL_ORDER if m in probs]
ids = labels.index; y = labels.values.astype(int)
rows = []
for M in CLS:
    p = probs[M].reindex(ids).values
    t = ST.classifier_table(y, p, {"0.5": 0.5, "operating": thr[M]} if M != "A" else {"0.5": 0.5}, B=B, seed=SEED)
    t.insert(0, "model", M); t.insert(2, "threshold_value", t.threshold.map({"0.5": 0.5, "operating": thr[M], "threshold-free": np.nan}))
    rows.append(t)
R1 = save_table(pd.concat(rows, ignore_index=True), "R1_classifiers")

idx = ST.boot_idx(len(y), B, SEED); rows = []
def paired(Ma, Mb):
    pa, pb = probs[Ma].reindex(ids).values, probs[Mb].reindex(ids).values
    for name, f in [("AUC", roc_auc_score), ("ECE", ST.ece_score)]:
        est = f(y, pa) - f(y, pb); bs = np.array([f(y[i], pa[i]) - f(y[i], pb[i]) for i in idx])
        rows.append(dict(comparison=f"{Ma} - {Mb}", metric=name, diff=est, ci_low=np.quantile(bs, .025), ci_high=np.quantile(bs, .975),
                         share_bootstrap_gt0=float((bs > 0).mean())))
for a, b_ in [("Anorm", "Arep"), ("Arep", "A"), ("B", "A"), ("B", "Anorm")]:
    if a in probs and b_ in probs: paired(a, b_)
R1b = save_table(pd.DataFrame(rows), "R1b_paired_classifier_differences")

def logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
rows = []
for M in [m for m in CLS if m in vals]:
    zv, yv = logit(vals[M].prob.values), vals[M].Target.values
    nll = lambda T: -np.mean(yv * np.log(1 / (1 + np.exp(-zv / T)) + 1e-12) + (1 - yv) * np.log(1 - 1 / (1 + np.exp(-zv / T)) + 1e-12))
    T = minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x
    from sklearn.linear_model import LogisticRegression
    pl = LogisticRegression(C=1e6).fit(zv[:, None], yv)          # Platt: slope and intercept on the logit
    pt = probs[M].reindex(ids).values; zt = logit(pt)
    pts = 1 / (1 + np.exp(-zt / T)); ptp = pl.predict_proba(zt[:, None])[:, 1]
    rows.append(dict(model=M, temperature=T, platt_slope=float(pl.coef_[0, 0]), platt_intercept=float(pl.intercept_[0]),
                     ece_test=ST.ece_score(y, pt), ece_test_temperature=ST.ece_score(y, pts), ece_test_platt=ST.ece_score(y, ptp),
                     brier_test=ST.brier(y, pt), brier_test_temperature=ST.brier(y, pts), brier_test_platt=ST.brier(y, ptp),
                     auc_test=roc_auc_score(y, pt), auc_test_platt=roc_auc_score(y, ptp)))
R1c = save_table(pd.DataFrame(rows), "R1c_temperature_scaling")
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 200):
    print(R1.pivot_table(index=["metric", "threshold"], columns="model", values="estimate").loc[:, CLS].to_string())
    print(); print(R1b.to_string(index=False)); print(); print(R1c.to_string(index=False))
for M in CLS:
    r = R1[(R1.model == M) & (R1.metric == "AUC")].iloc[0]; NUM[f"auc_{M}"] = [r.estimate, r.ci_low, r.ci_high]

### Figure RF1 · Discrimination and calibration of the classifiers

(a) ROC curves; (b) reliability diagrams (10 bins). Line style identifies the model.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.3)); dat_roc, dat_rel = [], []
for M in CLS:
    p = probs[M].reindex(ids).values; fpr, tpr, _ = roc_curve(y, p)
    ax[0].plot(fpr, tpr, ls=MODEL_LS[M], color=MODEL_INK[M], lw=1.6, label=f"{M} (AUC {roc_auc_score(y, p):.3f})")
    dat_roc.append(pd.DataFrame({"model": M, "fpr": fpr, "tpr": tpr}))
    rel = ST.reliability_curve(y, p, 10); rel.insert(0, "model", M); dat_rel.append(rel)
    ax[1].plot(rel.mean_pred, rel.frac_pos, ls=MODEL_LS[M], color=MODEL_INK[M], marker="o", ms=3.5, lw=1.4, label=f"{M} (ECE {ST.ece_score(y, p):.3f})")
for a in ax: a.plot([0, 1], [0, 1], color=S.NEUTRAL, lw=0.8, ls="--"); a.set_xlim(0, 1); a.set_ylim(0, 1.01)
ax[0].set(xlabel="False positive rate", ylabel="True positive rate"); ax[0].legend(loc="lower right", fontsize=7)
ax[1].set(xlabel="Mean predicted probability", ylabel="Observed fraction positive"); ax[1].legend(loc="upper left", fontsize=7)
S.panel_label(ax[0], "a"); S.panel_label(ax[1], "b"); fig.subplots_adjust(wspace=0.3)
save_fig(fig, "RF1_classifiers", {"roc": pd.concat(dat_roc), "reliability": pd.concat(dat_rel)})

### Figure RF2 · Effect of ImageNet normalisation on convergence and calibration (R1-d)

Validation AUC, loss and ECE per epoch for A-rep (no normalisation) and A-norm (ImageNet normalisation); same seed, data and recipe otherwise. Model B is shown for reference in the loss panel scale only through its own table (it uses a weighted loss).

In [ ]:
ep = {M: pd.read_csv(ROOT / "output" / "phase2" / f"train_{M}" / "epochs.csv").assign(model=M) for M in ["Arep", "Anorm"]
      if (ROOT / "output" / "phase2" / f"train_{M}" / "epochs.csv").exists()}
if len(ep) == 2:
    fig, ax = plt.subplots(1, 3, figsize=(7.2, 2.6))
    for M, e in ep.items():
        for a, col, lab in zip(ax, ["val_auc", "val_loss", "val_ece"], ["Validation AUC", "Validation loss (BCE)", "Validation ECE"]):
            a.plot(e.epoch, e[col], ls=MODEL_LS[M], color=MODEL_INK[M], marker="o", ms=4, lw=1.5, label=MODEL_LABELS[M]); a.set(xlabel="Epoch", ylabel=lab)
    for a, l in zip(ax, "abc"): a.set_xticks([1, 2, 3, 4, 5]); S.panel_label(a, l)
    ax[0].legend(loc="upper left", fontsize=7); fig.subplots_adjust(wspace=0.45)
    save_fig(fig, "RF2_normalisation_epochs", pd.concat(ep.values()))

## 2 · Grad-CAM vs Eigen-CAM, patient as the unit (A-03, D-15)

**R2**: for every model, overall (patient mean over the 9 conditions) and per condition: mean and median paired difference in XFI (Grad-CAM minus Eigen-CAM) with bootstrap CIs, share of patients with lower XFI for Grad-CAM, paired Wilcoxon, Holm over the 9 conditions, effect size r.
**R2b**: linear mixed model `xfi ~ method * type * level` with a random intercept per patient (statsmodels MixedLM, ML), fitted per model on Grad-CAM and Eigen-CAM rows; fixed effects with 95% CIs. Negative differences mean Grad-CAM is less fragile.

In [ ]:
def compare(a, b_, **kw):
    d = a - b_; me = ci(d); mdn = ci(d, np.median); w = ST.wilcoxon_paired(a, b_)
    return dict(**kw, n=len(d), mean_diff=me[0], mean_ci_low=me[1], mean_ci_high=me[2], median_diff=mdn[0], median_ci_low=mdn[1],
                median_ci_high=mdn[2], share_grad_lower=float((d < 0).mean()), p=w["p"], r=w["r"])
rows = []
for M in MODELS:
    x = main[M][main[M].Method.isin(METHODS)]
    pp = x.groupby(["Patient", "Method"]).xfi.mean().unstack(); rows.append(compare(pp.GradCAM.values, pp.EigenCAM.values, model=M, type="all", level="all"))
    sub = []
    for t in TYPES:
        for l in LEVELS:
            q = x[(x.Type == t) & (x.Level == l)].pivot_table(index="Patient", columns="Method", values="xfi")
            sub.append(compare(q.GradCAM.values, q.EigenCAM.values, model=M, type=t, level=l))
    ph = ST.holm([s["p"] for s in sub]); [s.update(p_holm=h) for s, h in zip(sub, ph)]; rows += sub
R2 = save_table(pd.DataFrame(rows), "R2_grad_vs_eigen")
with pd.option_context("display.float_format", "{:.4g}".format, "display.width", 200):
    print(R2[["model", "type", "level", "median_diff", "median_ci_low", "median_ci_high", "share_grad_lower", "p_holm", "r"]].to_string(index=False))
for M in MODELS:
    r = R2[(R2.model == M) & (R2.type == "all")].iloc[0]; NUM[f"grad_minus_eigen_overall_{M}"] = dict(median=[r.median_diff, r.median_ci_low, r.median_ci_high], p=r.p, r=r.r)

In [ ]:
import statsmodels.formula.api as smf, statsmodels.api as sm, warnings
rows = []
for M in [m for m in ["A", "B"] if m in MODELS]:
    x = main[M][main[M].Method.isin(METHODS)][["Patient", "Method", "Type", "Level", "xfi"]].copy()
    x["Method"] = pd.Categorical(x.Method, ["EigenCAM", "GradCAM"]); x["Type"] = pd.Categorical(x.Type, TYPES); x["Level"] = x.Level.astype(str)
    t0 = time.time(); mod = smf.mixedlm("xfi ~ Method * Type * Level", x, groups=x["Patient"]); fit, used = None, None
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for opt in ["cg", "bfgs", "powell"]:          # lbfgs hits a singular Hessian on this balanced design
            try:
                f_ = mod.fit(reml=True, method=opt, maxiter=2000)
                if f_.converged: fit, used = f_, opt; break
            except Exception as e:
                print(f"   {opt}: {type(e).__name__}")
        gee = smf.gee("xfi ~ Method * Type * Level", "Patient", x, cov_struct=sm.cov_struct.Exchangeable()).fit()
    assert fit is not None, "MixedLM did not converge with any optimiser"
    cf, gcf = fit.conf_int(), gee.conf_int()
    for k in fit.fe_params.index:
        rows.append(dict(model=M, term=k, estimate=fit.fe_params[k], ci_low=cf.loc[k, 0], ci_high=cf.loc[k, 1], p=fit.pvalues[k],
                         gee_estimate=gee.params[k], gee_ci_low=gcf.loc[k, 0], gee_ci_high=gcf.loc[k, 1], gee_p=gee.pvalues[k]))
    rows.append(dict(model=M, term="random intercept variance (patient)", estimate=float(fit.cov_re.iloc[0, 0])))
    rows.append(dict(model=M, term="residual variance", estimate=float(fit.scale)))
    rows.append(dict(model=M, term=f"optimiser={used}, REML, converged={fit.converged}; GEE exchangeable rho", estimate=float(gee.cov_struct.dep_params)))
    print(f"{M}: MixedLM ({used}, REML) converged in {time.time() - t0:.0f} s, n={len(x)}, patients={x.Patient.nunique()}")
R2b = save_table(pd.DataFrame(rows), "R2b_mixed_model")
with pd.option_context("display.float_format", "{:.4g}".format, "display.width", 200):
    print(R2b[R2b.term.str.contains("Method|variance")].to_string(index=False))

### Figure RF3 · Paired difference in XFI, Grad-CAM minus Eigen-CAM

One panel per model; rows are conditions plus the patient-level overall comparison. Points: median paired difference; bars: 95% bootstrap CI. Left of zero: Grad-CAM less fragile.

In [ ]:
fig, ax = plt.subplots(1, len(MODELS), figsize=(7.2 if len(MODELS) <= 2 else 9.6, 3.8), sharey=True)
ax = np.atleast_1d(ax); f3 = []
for a, M, letter in zip(ax, MODELS, "abcd"):
    d = R2[R2.model == M].copy()
    d["row"] = np.where(d.type == "all", "All conditions", d.type.map(S.TYPE_LABELS) + ", L" + d.level.astype(str))
    d = d.iloc[::-1].reset_index(drop=True); f3.append(d)
    for i, r in d.iterrows():
        allr = r.type == "all"
        a.errorbar(r.median_diff, i, xerr=[[r.median_diff - r.median_ci_low], [r.median_ci_high - r.median_diff]], fmt="D" if allr else "o",
                   ms=5, color=S.INK if allr else S.INK2, mfc=S.INK if allr else "white", capsize=2.5, lw=1.3)
    a.axvline(0, color=S.INK, lw=0.9); a.grid(axis="y", visible=False)
    a.set_yticks(range(len(d)), d.row); a.set_xlabel(f"{MODEL_LABELS[M]}\nmedian ΔXFI (Grad − Eigen)")
    S.panel_label(a, letter)
lim = max(abs(R2.median_ci_low).max(), abs(R2.median_ci_high).max()) * 1.1
for a in ax: a.set_xlim(-lim, lim)
fig.subplots_adjust(wspace=0.08)
save_fig(fig, "RF3_grad_vs_eigen", pd.concat(f3).drop(columns=["row"]))

## 3 · Degradation kinetics (A-13)

**R3**: mean XFI per model, method, type and level with bootstrap CI. **R4**: per-patient OLS slope of XFI on level (per type and pooled), mean with CI, Grad-CAM vs Eigen-CAM paired Wilcoxon with Holm.
The dashed line in RF4 is the cross-patient XFI of the same method and model: the typical dissimilarity between the clean explanations of two different patients. Values above it mean the perturbed explanation is less similar to its own clean explanation than another patient's explanation is.

In [ ]:
rows = []
for M in MODELS:
    x = main[M][main[M].Method.isin(METHODS)]
    for m in METHODS:
        for t in TYPES + ["pooled"]:
            for l in LEVELS:
                s = x[(x.Method == m) & (x.Level == l)]
                v = s[s.Type == t].xfi.values if t != "pooled" else s.groupby("Patient").xfi.mean().values
                e = ci(v); rows.append(dict(model=M, method=m, type=t, level=l, n=len(v), mean_xfi=e[0], ci_low=e[1], ci_high=e[2], median_xfi=float(np.median(v))))
R3 = save_table(pd.DataFrame(rows), "R3_xfi_by_condition")
rows = []
for M in MODELS:
    x = main[M][main[M].Method.isin(METHODS)]
    sl = pd.concat([ST.per_patient_slope(x, by=("Patient", "Method", "Type")),
                    ST.per_patient_slope(x.groupby(["Patient", "Method", "Level"], as_index=False).xfi.mean(), by=("Patient", "Method")).assign(Type="pooled")])
    sub = []
    for t in TYPES + ["pooled"]:
        q = sl[sl.Type == t].pivot_table(index="Patient", columns="Method", values="slope")
        rec = dict(model=M, type=t)
        for m in METHODS:
            e = ci(q[m].values); rec.update({f"{m}_slope": e[0], f"{m}_ci_low": e[1], f"{m}_ci_high": e[2]})
        d = ci((q.GradCAM - q.EigenCAM).values); w = ST.wilcoxon_paired(q.GradCAM.values, q.EigenCAM.values)
        rec.update(diff=d[0], diff_ci_low=d[1], diff_ci_high=d[2], p=w["p"], r=w["r"]); sub.append(rec)
    ph = ST.holm([s["p"] for s in sub]); [s.update(p_holm=h) for s, h in zip(sub, ph)]; rows += sub
R4 = save_table(pd.DataFrame(rows), "R4_slopes")
cross = {M: base[M][base[M].Method.str.startswith("CrossPatient")].groupby("Method").xfi.mean().rename(lambda k: k.split("_")[1]).to_dict() for M in MODELS}
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 200):
    print(R3[R3.type == "pooled"].pivot_table(index=["model", "method"], columns="level", values="mean_xfi").to_string())
    print(); print(R4.to_string(index=False)); print("\ncross-patient XFI:", cross)

### Figure RF4 · XFI against perturbation severity

Rows: models; columns: perturbation types. Mean XFI with 95% bootstrap band; dashed: cross-patient XFI of the same method (colour) and model.

In [ ]:
nr = len(MODELS); fig, ax = plt.subplots(nr, 3, figsize=(7.2, 2.3 * nr), sharey=True, squeeze=False); f4 = []
letters = iter("abcdefghijkl")
for i, M in enumerate(MODELS):
    for j, t in enumerate(TYPES):
        a = ax[i, j]
        for m in METHODS:
            s = R3[(R3.model == M) & (R3.method == m) & (R3.type == t)]
            a.fill_between(s.level, s.ci_low, s.ci_high, color=S.METHOD_COLORS[m], alpha=0.18, lw=0)
            a.plot(s.level, s.mean_xfi, marker=S.METHOD_MARKERS[m], color=S.METHOD_COLORS[m], label=S.METHOD_LABELS[m])
            a.axhline(cross[M][m], color=S.METHOD_COLORS[m], ls="--", lw=1)
        a.set_xticks(LEVELS, ["L1", "L2", "L3"]); a.grid(axis="x", visible=False)
        a.set_xlabel(f"{S.TYPE_LABELS[t]} ({M})"); S.panel_label(a, next(letters))
    ax[i, 0].set_ylabel("Mean XFI")
from matplotlib.lines import Line2D
ax[0, 0].set_ylim(0, None)
ax[0, 1].legend(handles=[Line2D([], [], color=S.METHOD_COLORS[m], marker=S.METHOD_MARKERS[m], label=S.METHOD_LABELS[m]) for m in METHODS] +
                [Line2D([], [], color=S.NEUTRAL, ls="--", lw=1, label="cross-patient XFI")], loc="upper center", fontsize=7)
fig.subplots_adjust(wspace=0.1, hspace=0.75)
save_fig(fig, "RF4_kinetics", {"xfi": R3[R3.type != "pooled"], "cross_patient": pd.DataFrame([dict(model=M, method=m, xfi=v) for M in cross for m, v in cross[M].items()])})

## 4 · Localization (R1-c, R1-BR1)

**R5**: clean-image IoU with the lesion box (Rel-90, Rel-80, Abs-0.5, Abs-0.7) by prediction-correctness group (all positives, TP, FN), per model and method.
**R6**: TP patients, Rel-90 IoU clean vs perturbed per condition; relative drop = 1 - mean(noisy)/mean(clean) with paired bootstrap CI.

In [ ]:
iou_cols = {"iou_rel90_c": "Rel-90", "iou_rel80_c": "Rel-80", "iou_abs5_c": "Abs-0.5", "iou_abs7_c": "Abs-0.7"}
rows = []
for M in MODELS:
    c = main[M][(main[M].Label == 1) & main[M].Method.isin(METHODS)].drop_duplicates(["Patient", "Method"])
    for m in METHODS:
        for g in ["all positives", "TP", "FN"]:
            s = c[c.Method == m] if g == "all positives" else c[(c.Method == m) & (c.grp == g)]
            for col, lab in iou_cols.items():
                e = ci(s[col].values); rows.append(dict(model=M, method=m, group=g, n=len(s), iou=lab, mean=e[0], ci_low=e[1], ci_high=e[2]))
R5 = save_table(pd.DataFrame(rows), "R5_clean_iou_by_correctness")
rows = []
for M in MODELS:
    tp = main[M][(main[M].grp == "TP") & main[M].Method.isin(METHODS)]
    for m in METHODS:
        for t in TYPES:
            for l in LEVELS:
                s = tp[(tp.Method == m) & (tp.Type == t) & (tp.Level == l)]
                ce, ne = ci(s.iou_rel90_c.values), ci(s.iou_rel90_n.values); rr = ST.ratio_of_means_ci(s.iou_rel90_n.values, s.iou_rel90_c.values, B=B, seed=SEED)
                rows.append(dict(model=M, method=m, type=t, level=l, n=len(s), clean=ce[0], clean_ci_low=ce[1], clean_ci_high=ce[2],
                                 noisy=ne[0], noisy_ci_low=ne[1], noisy_ci_high=ne[2], rel_drop=1 - rr[0], rel_drop_ci_low=1 - rr[2], rel_drop_ci_high=1 - rr[1]))
R6 = save_table(pd.DataFrame(rows), "R6_tp_iou_under_perturbation")
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
    print(R5[R5.iou == "Rel-90"].pivot_table(index=["model", "method"], columns="group", values="mean").to_string())
    print(); print(R6.pivot_table(index=["model", "method", "type"], columns="level", values="rel_drop").to_string())

### Figure RF5 · Localization of correctly classified positives under perturbation

Rows: models; columns: types. Rel-90 IoU with the lesion box; level 0 = clean image.

In [ ]:
fig, ax = plt.subplots(nr, 3, figsize=(7.2, 2.3 * nr), sharey=True, squeeze=False); letters = iter("abcdefghijkl")
for i, M in enumerate(MODELS):
    for j, t in enumerate(TYPES):
        a = ax[i, j]
        for m in METHODS:
            s = R6[(R6.model == M) & (R6.method == m) & (R6.type == t)].sort_values("level")
            xx = np.r_[0, s.level]; yy = np.r_[s.clean.iloc[0], s.noisy]
            lo = np.r_[s.clean_ci_low.iloc[0], s.noisy_ci_low]; hi = np.r_[s.clean_ci_high.iloc[0], s.noisy_ci_high]
            a.fill_between(xx, lo, hi, color=S.METHOD_COLORS[m], alpha=0.18, lw=0)
            a.plot(xx, yy, marker=S.METHOD_MARKERS[m], color=S.METHOD_COLORS[m], label=S.METHOD_LABELS[m])
        a.set_xticks([0, 1, 2, 3], ["clean", "L1", "L2", "L3"]); a.grid(axis="x", visible=False)
        a.set_xlabel(f"{S.TYPE_LABELS[t]} ({M})"); S.panel_label(a, next(letters))
    ax[i, 0].set_ylabel("IoU with lesion box")
ax[0, 0].set_ylim(0, None); ax[0, 1].legend(loc="lower center", fontsize=7)
fig.subplots_adjust(wspace=0.1, hspace=0.75)
save_fig(fig, "RF5_localization", R6)

## 5 · False activation with absolute thresholds (R2-M2, D-03)

Replaces the v1 percentile FAR (0.1 by construction, Phase 1 T8). For positives: share of background (non-box) pixels with normalised activation >= 0.5 (and >= 0.7), and share of heatmap energy outside the box. For negatives: share of the image with activation >= 0.5. Clean and perturbed, per condition.

In [ ]:
rows = []
for M in MODELS:
    x = main[M][main[M].Method.isin(METHODS)]
    for m in METHODS:
        for t in TYPES:
            for l in [0] + LEVELS:
                s = x[(x.Method == m) & (x.Type == t) & (x.Level == max(l, 1))]
                pos, neg = s[(s.Label == 1) & (s.degenerate == 0)], s[(s.Label == 0) & (s.degenerate == 0)]
                sfx = "_c" if l == 0 else "_n"
                rec = dict(model=M, method=m, type=t, level=l)
                for col in ["far_bg_abs5", "far_bg_abs7", "energy_out_box"]:
                    e = ci(pos[col + sfx].values); rec.update({f"{col}_pos": e[0], f"{col}_pos_ci_low": e[1], f"{col}_pos_ci_high": e[2]})
                e = ci(neg["act_area_abs5" + sfx].values); rec.update(act_area_abs5_neg=e[0], act_area_abs5_neg_ci_low=e[1], act_area_abs5_neg_ci_high=e[2])
                rows.append(rec)
R7 = save_table(pd.DataFrame(rows), "R7_absolute_false_activation")
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
    print(R7.pivot_table(index=["model", "method", "type"], columns="level", values="far_bg_abs5_pos").to_string())

### Figure RF6 · Background activation of positive images (activation >= 0.5 outside the lesion box)

Rows: models; columns: types; level 0 = clean image. Unlike the percentile FAR, this measure responds to the perturbation.

In [ ]:
fig, ax = plt.subplots(nr, 3, figsize=(7.2, 2.3 * nr), sharey=True, squeeze=False); letters = iter("abcdefghijkl")
for i, M in enumerate(MODELS):
    for j, t in enumerate(TYPES):
        a = ax[i, j]
        for m in METHODS:
            s = R7[(R7.model == M) & (R7.method == m) & (R7.type == t)].sort_values("level")
            a.fill_between(s.level, s.far_bg_abs5_pos_ci_low, s.far_bg_abs5_pos_ci_high, color=S.METHOD_COLORS[m], alpha=0.18, lw=0)
            a.plot(s.level, s.far_bg_abs5_pos, marker=S.METHOD_MARKERS[m], color=S.METHOD_COLORS[m], label=S.METHOD_LABELS[m])
        a.set_xticks([0, 1, 2, 3], ["clean", "L1", "L2", "L3"]); a.grid(axis="x", visible=False)
        a.set_xlabel(f"{S.TYPE_LABELS[t]} ({M})"); S.panel_label(a, next(letters))
    ax[i, 0].set_ylabel("Background share ≥ 0.5")
ax[0, 0].set_ylim(0, None); ax[0, 1].legend(loc="lower center", fontsize=7)
fig.subplots_adjust(wspace=0.1, hspace=0.75)
save_fig(fig, "RF6_absolute_far", R7)

## 6 · What XFI measures: validity and component ablation (A-04, A-05, D-17)

**R8**: Spearman correlation of XFI with input-image PSNR and SSIM (degradation of the input, not of the heatmap; non-circular), with the absolute change in localization (|delta IoU|, positives) and with prediction flips (point-biserial), per model and method.
**R9** (component ablation): Spearman correlation of XFI with each of its components, and agreement of the per-condition Grad-CAM vs Eigen-CAM conclusion (sign of the median paired difference) when the comparison uses a single component (1 - SSIM, 1 - normalised Pearson, 1 - normalised Spearman) instead of XFI.

In [ ]:
from scipy.stats import spearmanr, pointbiserialr
rows = []
for M in MODELS:
    x = main[M][main[M].Method.isin(METHODS)]
    for m in METHODS:
        s = x[x.Method == m]; sp = s[s.Label == 1]
        rows.append(dict(model=M, method=m, rho_input_psnr=spearmanr(s.xfi, s.input_psnr)[0], rho_input_ssim=spearmanr(s.xfi, s.input_ssim)[0],
                         rho_abs_d_iou_pos=spearmanr(sp.xfi, sp.abs_d_iou, nan_policy="omit")[0], r_flip=pointbiserialr(s.flip, s.xfi)[0],
                         rho_clean_iou_pos=spearmanr(sp.xfi, sp.iou_rel90_c, nan_policy="omit")[0]))
R8 = save_table(pd.DataFrame(rows), "R8_validity")
rows = []
for M in MODELS:
    x = main[M][main[M].Method.isin(METHODS)].copy()
    x["c_ssim"] = 1 - x.ssim; x["c_pearson"] = 1 - (x.p_corr.fillna(0) + 1) / 2; x["c_spearman"] = 1 - (x.s_corr.fillna(0) + 1) / 2
    rec = dict(model=M, **{f"rho_xfi_{c}": spearmanr(x.xfi, x[c])[0] for c in ["c_ssim", "c_pearson", "c_spearman"]})
    for c in ["c_ssim", "c_pearson", "c_spearman"]:
        agree = 0
        for t in TYPES:
            for l in LEVELS:
                q = x[(x.Type == t) & (x.Level == l)]
                a_ = q.pivot_table(index="Patient", columns="Method", values="xfi"); b_ = q.pivot_table(index="Patient", columns="Method", values=c)
                agree += int(np.sign(np.median(a_.GradCAM - a_.EigenCAM)) == np.sign(np.median(b_.GradCAM - b_.EigenCAM)))
        rec[f"agree_{c}_of_9"] = agree
    rows.append(rec)
R9 = save_table(pd.DataFrame(rows), "R9_component_ablation")
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
    print(R8.to_string(index=False)); print(); print(R9.to_string(index=False))

### Figure RF7 · XFI follows the degradation of the input and the change in localization

(a) Mean XFI by input-PSNR bin (all conditions pooled); (b) mean |delta IoU| by XFI decile (positive patients). Colour: method; line style: model.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.0)); d7a, d7b = [], []
bins = np.arange(9, 43, 3)
for M in MODELS:
    x = main[M][main[M].Method.isin(METHODS)]
    for m in METHODS:
        s = x[x.Method == m].copy(); s["bin"] = pd.cut(s.input_psnr, bins)
        g = s.groupby("bin", observed=True).agg(xfi=("xfi", "mean"), n=("xfi", "size"), psnr=("input_psnr", "mean")).reset_index(); g = g[g.n >= 50]
        ax[0].plot(g.psnr, g.xfi, ls=MODEL_LS[M], color=S.METHOD_COLORS[m], marker=S.METHOD_MARKERS[m], ms=3.5, lw=1.4, label=f"{S.METHOD_LABELS[m]}, {M}")
        d7a.append(g.assign(model=M, method=m).drop(columns="bin"))
        sp = s[s.Label == 1].copy(); sp["dec"] = pd.qcut(sp.xfi, 10, labels=False, duplicates="drop")
        h = sp.groupby("dec").agg(abs_d_iou=("abs_d_iou", "mean"), xfi=("xfi", "mean")).reset_index()
        ax[1].plot(h.dec + 1, h.abs_d_iou, ls=MODEL_LS[M], color=S.METHOD_COLORS[m], marker=S.METHOD_MARKERS[m], ms=3.5, lw=1.4)
        d7b.append(h.assign(model=M, method=m))
ax[0].set(xlabel="Input PSNR, clean vs perturbed (dB)", ylabel="Mean XFI", ylim=(0, None)); ax[0].legend(fontsize=6.5, loc="upper right")
ax[1].set(xlabel="XFI decile (1 = least fragile)", ylabel="Mean |ΔIoU| (Rel-90)", ylim=(0, None)); ax[1].set_xticks(range(1, 11))
S.panel_label(ax[0], "a"); S.panel_label(ax[1], "b"); fig.subplots_adjust(wspace=0.3)
save_fig(fig, "RF7_validity", {"psnr_bins": pd.concat(d7a), "xfi_deciles": pd.concat(d7b)})

## 7 · Eigen-CAM sign ablation (D-11)

**R10**: per model and condition, the share of Eigen-CAM projections where the ReLU keeps the minor lobe (clean, perturbed), the share of sign flips between clean and perturbed (signed correlation < 0), and mean XFI of Eigen-CAM with and without sign correction.

In [ ]:
rows = []
for M in MODELS:
    e = main[M][main[M].Method == "EigenCAM"]; sc = main[M][main[M].Method == "EigenCAM_SC"].set_index(["Patient", "Type", "Level"]).xfi
    for t in TYPES:
        for l in LEVELS:
            s = e[(e.Type == t) & (e.Level == l)]
            x1 = ci(s.xfi.values); x2 = ci(sc.xs((t, l), level=["Type", "Level"]).values)
            rows.append(dict(model=M, type=t, level=l, minor_lobe_clean=s.eig_minor_lobe_c.mean(), minor_lobe_noisy=s.eig_minor_lobe_n.mean(),
                             sign_flip=(s.eig_signed_corr < 0).mean(), xfi_eigen=x1[0], xfi_eigen_ci_low=x1[1], xfi_eigen_ci_high=x1[2],
                             xfi_eigen_sc=x2[0], xfi_eigen_sc_ci_low=x2[1], xfi_eigen_sc_ci_high=x2[2]))
R10 = save_table(pd.DataFrame(rows), "R10_eigen_sign")
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
    print(R10.to_string(index=False))

### Figure RF8 · Eigen-CAM sign behaviour

(a) Share of clean-to-perturbed sign flips of the Eigen-CAM projection per condition; (b) mean Eigen-CAM XFI without and with sign correction. Line style: model.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.0)); xl = [f"{t[:4]}\nL{l}" for t in TYPES for l in LEVELS]; xx = np.arange(9)
def seg(a, y_, **kw):
    """one line segment per perturbation type, so that types are not joined"""
    y_ = np.asarray(y_)
    for k in range(3):
        a.plot(xx[3 * k:3 * k + 3], y_[3 * k:3 * k + 3], **(kw if k == 0 else {**kw, "label": None}))
for M in MODELS:
    s = R10[R10.model == M]
    seg(ax[0], s.sign_flip, ls=MODEL_LS[M], color=S.INK, marker="o", ms=3.5, lw=1.3, label=MODEL_LABELS[M])
    seg(ax[1], s.xfi_eigen, ls=MODEL_LS[M], color=S.METHOD_COLORS["EigenCAM"], marker="s", ms=3.5, lw=1.3, label=f"Eigen-CAM, {M}")
    seg(ax[1], s.xfi_eigen_sc, ls=MODEL_LS[M], color=S.NEUTRAL, marker="^", ms=3.5, lw=1.3, label=f"sign-corrected, {M}")
for a in ax:
    a.set_xticks(xx, xl, fontsize=7); a.grid(axis="x", visible=False)
    for v in (2.5, 5.5): a.axvline(v, color=S.GRID, lw=0.8)
ax[0].set(ylabel="Share of sign flips", ylim=(0, 0.55)); ax[0].legend(fontsize=7, loc="upper left")
ax[1].set(ylabel="Mean XFI", ylim=(0, 0.85)); ax[1].legend(fontsize=6.5, loc="upper center", ncol=2)
S.panel_label(ax[0], "a"); S.panel_label(ax[1], "b"); fig.subplots_adjust(wspace=0.3)
save_fig(fig, "RF8_eigen_sign", R10)

## 8 · Baselines (D-14)

**R11**: mean XFI with CI of each baseline per model: independent pixel-noise maps, smooth random maps (7 x 7 upsampled), cross-patient pairs of clean explanations, explanations of a model whose top block (layer4 + fc) was re-initialised (sanity check), and the uniform map (convention). The degenerate column gives the share of pairs where one map was constant.

In [ ]:
rows = []
for M in MODELS:
    b_ = base[M]
    for meth, g in b_.groupby("Method"):
        pv = g.groupby("Patient").xfi.mean().values; e = ci(pv)
        rows.append(dict(model=M, baseline=meth, n_patients=len(pv), mean_xfi=e[0], ci_low=e[1], ci_high=e[2], median_xfi=float(np.median(pv)), degenerate=g.degenerate.mean()))
    for m in METHODS:
        for l in LEVELS:
            pv = main[M][(main[M].Method == m) & (main[M].Level == l)].groupby("Patient").xfi.mean().values; e = ci(pv)
            rows.append(dict(model=M, baseline=f"{m} perturbed L{l} (reference)", n_patients=len(pv), mean_xfi=e[0], ci_low=e[1], ci_high=e[2], median_xfi=float(np.median(pv))))
R11 = save_table(pd.DataFrame(rows), "R11_baselines")
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 200):
    print(R11.pivot_table(index="baseline", columns="model", values="mean_xfi").to_string())
for M in MODELS:
    NUM[f"baselines_{M}"] = R11[R11.model == M].set_index("baseline").mean_xfi.round(4).to_dict()

### Figure RF9 · XFI of the explanation methods against the baselines

Per model: distribution of patient-level XFI (mean over the three perturbation types) per level and method, with the mean of each baseline as a horizontal line.

In [ ]:
fig, ax = plt.subplots(1, len(MODELS), figsize=(7.2 if len(MODELS) <= 2 else 9.6, 3.2), sharey=True); ax = np.atleast_1d(ax); d9 = []
bl_style = {"CrossPatient": ("--", "cross-patient"), "ModelRandom": (":", "randomised model"), "Random_Smooth": ("-.", "smooth random"), "Random_Pixel": ("-", "pixel random")}
for a, M, letter in zip(ax, MODELS, "abcd"):
    pos = 0; ticks, tl = [], []
    for l in LEVELS:
        for m in METHODS:
            v = main[M][(main[M].Method == m) & (main[M].Level == l)].groupby("Patient").xfi.mean().values
            bp = a.boxplot(v, positions=[pos], widths=0.6, showfliers=False, patch_artist=True)
            for k in ["boxes"]: [p.set(facecolor="white", edgecolor=S.METHOD_COLORS[m], lw=1.3) for p in bp[k]]
            for k in ["whiskers", "caps", "medians"]: [p.set(color=S.METHOD_COLORS[m], lw=1.2) for p in bp[k]]
            d9.append(pd.DataFrame({"model": M, "method": m, "level": l, "xfi_patient_mean": v})); pos += 1
        ticks.append(pos - 1.5); tl.append(f"L{l}"); pos += 0.6
    for key, (ls, lab) in bl_style.items():
        v = R11[(R11.model == M) & R11.baseline.str.startswith(key)].mean_xfi.mean()
        a.axhline(v, color=S.NEUTRAL, ls=ls, lw=1.1, label=lab)
    a.set_xticks(ticks, tl); a.grid(axis="x", visible=False); a.set_xlabel(MODEL_LABELS[M]); S.panel_label(a, letter)
ax[0].set_ylabel("Patient-level XFI"); ax[0].set_ylim(0, 0.8)
from matplotlib.lines import Line2D
h = [Line2D([], [], color=S.METHOD_COLORS[m], lw=2, label=S.METHOD_LABELS[m]) for m in METHODS] + ax[0].get_legend_handles_labels()[0]
ax[0].legend(handles=h, fontsize=6.5, loc="upper left", ncol=2, frameon=True, facecolor="white", edgecolor="none", framealpha=1)
fig.subplots_adjust(wspace=0.08)
save_fig(fig, "RF9_baselines", {"methods": pd.concat(d9), "baselines": R11})

## 9 · Prediction flips (A-11)

**R12**: share of patients whose predicted class changes under each perturbation, per model; and mean XFI of rows with and without a flip (Mann-Whitney U, per model and method, all conditions pooled).

In [ ]:
from scipy.stats import mannwhitneyu
rows = []
for M in MODELS:
    g = main[M][main[M].Method == "GradCAM"]
    for t in TYPES:
        for l in LEVELS:
            s = g[(g.Type == t) & (g.Level == l)]; e = ci(s.flip.values.astype(float))
            rows.append(dict(model=M, kind="flip_rate", type=t, level=l, value=e[0], ci_low=e[1], ci_high=e[2]))
    for m in METHODS:
        s = main[M][main[M].Method == m]; f1, f0 = s[s.flip == 1].xfi, s[s.flip == 0].xfi
        rows.append(dict(model=M, kind=f"xfi_flip_vs_noflip_{m}", type="all", level="all", value=f1.mean() - f0.mean(),
                         xfi_flip=f1.mean(), xfi_noflip=f0.mean(), n_flip=len(f1), p=mannwhitneyu(f1, f0).pvalue if len(f1) else np.nan))
R12 = save_table(pd.DataFrame(rows), "R12_flips")
with pd.option_context("display.float_format", "{:.4g}".format, "display.width", 200):
    print(R12[R12.kind == "flip_rate"].pivot_table(index=["model", "type"], columns="level", values="value").to_string())
    print(R12[R12.kind != "flip_rate"].to_string(index=False))

### Figure RF10 · Prediction flips under perturbation

(a) Share of patients whose predicted class changes, per condition (line style: model); (b) mean XFI of perturbed images with and without a prediction flip.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.0)); fr = R12[R12.kind == "flip_rate"]
for M in MODELS:
    s = fr[fr.model == M]; seg(ax[0], s.value, ls=MODEL_LS[M], color=S.INK, marker="o", ms=3.5, lw=1.3, label=MODEL_LABELS[M])
    for k in range(3):
        ax[0].fill_between(xx[3 * k:3 * k + 3], s.ci_low.values[3 * k:3 * k + 3], s.ci_high.values[3 * k:3 * k + 3], color=S.NEUTRAL, alpha=0.15, lw=0)
for v in (2.5, 5.5): ax[0].axvline(v, color=S.GRID, lw=0.8)
ax[0].set_xticks(xx, xl, fontsize=7); ax[0].set(ylabel="Share of prediction flips", ylim=(0, 0.5)); ax[0].legend(fontsize=7, loc="upper right"); ax[0].grid(axis="x", visible=False)
w = 0.18; ff = R12[R12.kind != "flip_rate"].copy(); ff["method"] = ff.kind.str.replace("xfi_flip_vs_noflip_", "")
for i, M in enumerate(MODELS):
    for j, m in enumerate(METHODS):
        r = ff[(ff.model == M) & (ff.method == m)].iloc[0]; x0 = i + (j - 0.5) * 2 * w
        ax[1].bar([x0 - w / 2, x0 + w / 2], [r.xfi_noflip, r.xfi_flip], width=w * 0.95, color=["white", S.METHOD_COLORS[m]], edgecolor=S.METHOD_COLORS[m], lw=1.2)
ax[1].set_xticks(range(len(MODELS)), MODELS); ax[1].set(ylabel="Mean XFI", ylim=(0, 0.6)); ax[1].grid(axis="x", visible=False)
from matplotlib.patches import Patch
ax[1].legend(handles=[Patch(facecolor="white", edgecolor=S.INK, label="no flip"), Patch(facecolor=S.NEUTRAL, edgecolor=S.INK, label="flip")] +
             [Patch(facecolor=S.METHOD_COLORS[m], label=S.METHOD_LABELS[m]) for m in METHODS], fontsize=7, loc="upper left", ncol=2)
S.panel_label(ax[0], "a"); S.panel_label(ax[1], "b"); fig.subplots_adjust(wspace=0.3)
save_fig(fig, "RF10_flips", R12)

## 10 · Consistency across classifiers (R1-V1, R1-V2) and numbers for the manuscript

**R13**: per condition and method, mean XFI for each model and the correlation of the condition-level pattern between models. `numbers_v2.json` collects the headline numbers that the manuscript will quote; Phase 9 checks the text against it.

In [ ]:
r3 = R3[R3.type != "pooled"].pivot_table(index=["method", "type", "level"], columns="model", values="mean_xfi")[MODELS]
R13 = save_table(r3.reset_index(), "R13_condition_pattern_by_model")
print(r3.round(3).to_string()); print("\nSpearman correlation of the 18 condition means between models:"); print(r3.corr("spearman").round(3).to_string())
sign = R2[R2.type != "all"].pivot_table(index=["type", "level"], columns="model", values="median_diff")
print("\nsign of median(Grad - Eigen) per condition:"); print(np.sign(sign).astype(int).to_string())
for M in MODELS:
    NUM[f"mean_xfi_{M}"] = main[M][main[M].Method.isin(METHODS + ["EigenCAM_SC"])].groupby(["Patient", "Method"]).xfi.mean().groupby("Method").mean().round(4).to_dict()
NUM["condition_pattern_spearman"] = r3.corr("spearman").round(4).to_dict()
(OUT / "numbers_v2.json").write_text(json.dumps(NUM, indent=2, default=float)); PRODUCED.append(OUT / "numbers_v2.json")

## Summary, artifacts and LOG entry

In [ ]:
from datetime import date
arts = pd.DataFrame([dict(path=str(Path(p).relative_to(ROOT)), bytes=Path(p).stat().st_size, sha256=sha256(p)) for p in PRODUCED])
arts.to_csv(OUT / "artifacts_phase4.csv", index=False)
print(f"{len(arts)} files ({arts.bytes.sum() / 1e6:.1f} MB)")
print("\n--- ARTIFACTS line ---")
print(f"ART-020,{date.today()},output/phase4/artifacts_phase4.csv,table,4,notebooks/03_analysis.ipynb,{sha256(OUT / 'artifacts_phase4.csv')},"
      f"Phase 4 analysis of v2 ({', '.join(MODELS)}): tables R1-R13, figures RF1-RF10, numbers_v2.json")
print(f"\n--- LOG entry ---\n## {date.today()} | Phase 4 analysis (03_analysis), models {MODELS}")
for M in MODELS:
    r = R2[(R2.model == M) & (R2.type == "all")].iloc[0]
    print(f"- {M}: AUC {NUM[f'auc_{M}'][0]:.3f}; mean XFI {NUM[f'mean_xfi_{M}']}; Grad-Eigen median {r.median_diff:+.4f} "
          f"[{r.median_ci_low:+.4f}, {r.median_ci_high:+.4f}] p {r.p:.2g}; cross-patient {cross[M]}")